In [1]:
!pip install -q pymupdf pdfplumber pandas numpy scikit-learn
!pip install -q sentence-transformers chromadb rank-bm25
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community
!pip install -q matplotlib seaborn

In [2]:
import torch

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)
else:
    print("GPU not detected")

CUDA Available: True
GPU Name: NVIDIA GeForce RTX 2060
CUDA Version: 12.6


In [7]:
import os
import random

DATA_PATH = r"F:\research paper"

pdf_files = [f for f in os.listdir(DATA_PATH) if f.endswith(".pdf")]

# max 3 papers
pdf_files = pdf_files[:3]

random.shuffle(pdf_files)

paper_dict = {f"P{i+1}": os.path.join(DATA_PATH, f) for i, f in enumerate(pdf_files)}

paper_dict

{'P1': 'F:\\research paper\\Design and implementation of deep learning-based framework.pdf'}

In [8]:
import fitz  # PyMuPDF

documents = []

for paper_id, path in paper_dict.items():
    doc = fitz.open(path)
    
    for page_num, page in enumerate(doc):
        text = page.get_text()
        
        documents.append({
            "paper_id": paper_id,
            "page": page_num + 1,
            "text": text
        })

# preview
documents[:2]

[{'paper_id': 'P1',
  'page': 1,
  'text': 'Design and implementation of deep learning-based framework for \nmulti-class fault diagnosis in complex chemical process systems\nRemigius Nnadozie Ewuzie , Shivaneswar Gunasekaran\n, Zainal Ahmad , Norazwan Md Nor *\nSchool of Chemical Engineering, Engineering Campus, Universiti Sains Malaysia, 14300 Nibong Tebal, Pulau Pinang, Malaysia\nA R T I C L E  I N F O\nKeywords:\nProcess monitoring\nMachine learning\nDeep learning\nMulti-class classification\nPerformance comparison\nFault diagnosis\nA B S T R A C T\nFault diagnosis in modern chemical plants is increasingly challenging due to process complexity, nonlinearity, \nand high-risk operations, where undetected faults can cause severe safety and economic consequences. Con\xad\nventional machine learning (ML) models suffer from reliance on handcrafted features, poor generalization in \nhigh-dimensional spaces, and limited labeled data, resulting in reduced diagnostic performance. To overcome 

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = []

for doc in documents:
    split_texts = splitter.split_text(doc["text"])
    
    for chunk in split_texts:
        chunks.append({
            "paper_id": doc["paper_id"],
            "page": doc["page"],
            "text": chunk
        })

len(chunks)

331

In [11]:
from sentence_transformers import SentenceTransformer
import chromadb
from rank_bm25 import BM25Okapi

# 1. Load embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Prepare texts
texts = [c["text"] for c in chunks]

# 3. Create embeddings
embeddings = embed_model.encode(texts, show_progress_bar=True)

# 4. Setup ChromaDB
client = chromadb.Client()
collection = client.create_collection(name="rag_collection")

# 5. Store in vector DB
for i, chunk in enumerate(chunks):
    collection.add(
        ids=[str(i)],
        embeddings=[embeddings[i]],
        documents=[chunk["text"]],
        metadatas=[{
            "paper_id": chunk["paper_id"],
            "page": chunk["page"]
        }]
    )

# 6. Setup BM25
tokenized_corpus = [text.split() for text in texts]
bm25 = BM25Okapi(tokenized_corpus)

print("✅ Embedding + BM25 ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

✅ Embedding + BM25 ready


In [12]:
import numpy as np

def hybrid_search(query, top_k=5):
    
    # --- BM25 ---
    tokenized_query = query.split()
    bm25_scores = bm25.get_scores(tokenized_query)
    
    # --- Vector ---
    query_embedding = embed_model.encode([query])[0]
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    
    vector_ids = results["ids"][0]
    
    # --- Combine scores ---
    combined = []
    
    for i in range(len(chunks)):
        score = bm25_scores[i]
        
        if str(i) in vector_ids:
            score += 1.0  # boost if in vector result
        
        combined.append((i, score))
    
    # sort
    combined = sorted(combined, key=lambda x: x[1], reverse=True)
    
    # top results
    final = []
    for idx, _ in combined[:top_k]:
        final.append(chunks[idx])
    
    return final


# test
hybrid_search("deep learning model for classification")

[{'paper_id': 'P1',
  'page': 14,
  'text': 'formance of deep learning models for process monitoring and fault \ndiagnosis, particularly in complex industrial systems such as the Ten\xad\nnessee Eastman process. The TEP dataset is characterized by high \ndimensionality, nonlinear dynamics, and strong interdependencies \namong process variables, which necessitate careful model configuration \nto ensure reliable and robust classification results. In this study, two key \nhyperparameters, the number of layers (2, 3, 4, 5) and the dropout rate'},
 {'paper_id': 'P1',
  'page': 17,
  'text': 'classes. Among the deep learning models, the convolutional neural \nnetwork (CNN) achieved the highest classification accuracy, and its \nconfusion matrix is presented in Fig. 14. The matrix reveals a more \nconsistent and uniformly dominant diagonal compared to the ML model, \nreflecting improved per-class classification accuracy. This enhanced \ndiagonal dominance demonstrates the DL model’s superior 

In [13]:
import requests

def generate_answer(query, retrieved_chunks):
    
    context = "\n\n".join([c["text"] for c in retrieved_chunks])
    
    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3",
            "prompt": prompt,
            "stream": False
        }
    )
    
    return response.json()["response"]


# test
query = "What model is used in the paper?"
retrieved = hybrid_search(query)
generate_answer(query, retrieved)

'The model used in the paper is a Convolutional Neural Network (CNN). Additionally, an Autoencoder (AE) is also mentioned as part of the deep learning-based framework.'

In [14]:
def structured_analysis(task):
    
    prompt = f"""
You are analyzing multiple research papers (P1, P2, P3).

Task: {task}

Return output STRICTLY in table format:

Paper | {task}

Instructions:
- One row per paper (P1, P2, P3)
- Keep it concise
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3",
            "prompt": prompt,
            "stream": False
        }
    )
    
    return response.json()["response"]


# GAP
def get_gap():
    retrieved = hybrid_search("limitations research gap")
    return structured_analysis("Research Gap")

# Methodology
def get_methodology():
    retrieved = hybrid_search("methodology model dataset training")
    return structured_analysis("Methodology")

# Results
def get_results():
    retrieved = hybrid_search("results accuracy precision recall")
    return structured_analysis("Results")


# test
print(get_gap())

Here is the output in table format:

| Paper | Research Gap |
| --- | --- |
| P1 | Limited exploration of [specific methodology] for real-world applications. |
| P2 | Insufficient attention to [particular population]'s needs and concerns. |
| P3 | Failure to consider [overlooked factor] in [specific context]. |

Let me know if you'd like me to expand on the research gaps or provide further analysis!


In [24]:
!pip install streamlit